# 🧪 实验二：MobileNetV3 网络结构解析

**MobileNetV3 图像分类实战教程**

---

## 实验目标

1. 理解 MobileNetV3 的核心设计思想（深度可分离卷积、SE 模块、h-swish 激活函数）
2. 逐模块实现并运行 MobileNetV3 的各个组件
3. 可视化 MobileNetV3-Large 和 Small 的网络结构
4. 统计模型参数量和计算量

---

In [ ]:
# ====== 1. 导入所需库 ======
import torch
import torch.nn as nn
import torch.nn.functional as F

# 导入 torch_npu（Ascend NPU 支持）
import torch_npu

print(f"PyTorch 版本: {torch.__version__}")
print(f"NPU 可用: {torch.npu.is_available()}")

import warnings
warnings.filterwarnings("ignore", message=".*owner does not match.*")
warnings.filterwarnings("ignore", message=".*TASK_QUEUE_ENABLE.*")
warnings.filterwarnings("ignore", message=".*Permission mismatch.*")

---
## 2. MobileNetV3 设计思想

MobileNetV3 是 Google 在 2019 年提出的高效轻量级网络，融合了以下关键技术：

### 2.1 深度可分离卷积 (Depthwise Separable Convolution)

将标准卷积分解为两步：
- **Depthwise Conv**: 每个输入通道单独用一个卷积核处理
- **Pointwise Conv**: 用 1×1 卷积组合各通道输出

计算量对比：
- 标准卷积：$D_K \times D_K \times C_{in} \times C_{out} \times H \times W$
- 深度可分离：$D_K \times D_K \times C_{in} \times H \times W + C_{in} \times C_{out} \times H \times W$
- 计算量比值：$\frac{1}{C_{out}} + \frac{1}{D_K^2}$

当 $D_K=3$ 时，计算量约为标准卷积的 **1/8 ~ 1/9**！

### 2.2 SE 模块 (Squeeze-and-Excitation)

通过学习通道间的依赖关系，自适应地重新校准通道特征响应：
1. **Squeeze**: 全局平均池化，压缩空间信息
2. **Excitation**: 两个全连接层学习通道权重
3. **Scale**: 将权重乘以原始特征图

### 2.3 h-swish 激活函数

$$\text{h-swish}(x) = x \cdot \frac{\text{ReLU6}(x+3)}{6}$$

相比 swish，h-swish 在移动设备上推理更快，因为去除了 sigmoid 计算。

### 2.4 网络结构搜索 (NAS)

MobileNetV3 使用 Neural Architecture Search (NAS) 和 NetAdapt 算法自动搜索最优网络结构。

---
## 3. 逐模块代码解析

MobileNetV3 的核心组件包括：
1. **h_sigmoid / h_swish** — 高效的硬式激活函数
2. **SqueezeBlock** — SE 注意力模块
3. **MobileBlock** — 基础构建块（深度可分离卷积 + SE + 残差连接）
4. **MobileNetV3** — 完整网络（Large / Small）

让我们逐一实现并分析这些模块。

In [ ]:
# ====== 2. h_sigmoid 与 h_swish 激活函数 ======

class h_sigmoid(nn.Module):
    """
    硬 Sigmoid 激活函数
    h_sigmoid(x) = ReLU6(x + 3) / 6
    比标准 sigmoid 计算更快，适合移动端部署
    """
    def __init__(self, inplace=True):
        super(h_sigmoid, self).__init__()
        self.inplace = inplace

    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.


class h_swish(nn.Module):
    """
    硬 Swish 激活函数
    h_swish(x) = x * ReLU6(x + 3) / 6
    """
    def __init__(self, inplace=True):
        super(h_swish, self).__init__()
        self.inplace = inplace

    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x


# 可视化激活函数
import matplotlib.pyplot as plt
import numpy as np

x = torch.linspace(-6, 6, 200)
h_sig = h_sigmoid(inplace=False)
h_sw = h_swish(inplace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x.numpy(), h_sig(x).numpy(), 'b-', linewidth=2)
axes[0].plot(x.numpy(), torch.sigmoid(x).numpy(), 'r--', linewidth=2, alpha=0.7)
axes[0].set_title('Activation Function Comparison')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend(['h_sigmoid', 'sigmoid'])
axes[0].grid(True, alpha=0.3)

axes[1].plot(x.numpy(), h_sw(x).numpy(), 'b-', linewidth=2)
axes[1].plot(x.numpy(), x.numpy() * torch.sigmoid(x).numpy(), 'r--', linewidth=2, alpha=0.7)
axes[1].set_title('Activation Function Comparison')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend(['h_swish', 'swish'])
axes[1].grid(True, alpha=0.3)

# ReLU6
axes[2].plot(x.numpy(), F.relu6(x).numpy(), 'g-', linewidth=2)
axes[2].plot(x.numpy(), F.relu(x).numpy(), 'm--', linewidth=2, alpha=0.7)
axes[2].set_title('Activation Function Comparison')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
axes[2].legend(['ReLU6', 'ReLU'])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 h_sigmoid 和 h_swish 在 [-3, 3] 区间内与标准版本接近，但计算更快。"
      "ReLU6 将输出限制在 0~6 之间，有利于量化部署。")

In [ ]:
# ====== 3. SqueezeBlock (SE 模块) ======

class SqueezeBlock(nn.Module):
    """
    Squeeze-and-Excitation 模块
    
    结构:
        Global AvgPool → Linear(exp_size, exp_size//4) → ReLU 
        → Linear(exp_size//4, exp_size) → h_sigmoid → Scale
    """
    def __init__(self, exp_size, divide=4):
        super(SqueezeBlock, self).__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )

    def forward(self, x):
        batch, channels, height, width = x.size()
        # Squeeze: 全局平均池化
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        # Excitation: 学习通道权重
        out = self.dense(out)
        out = out.view(batch, channels, 1, 1)
        # Scale: 加权原始特征
        return out * x


# 测试 SE 模块
se = SqueezeBlock(exp_size=24)
test_input = torch.randn(2, 24, 14, 14)
test_output = se(test_input)
print(f"SE 模块输入形状: {test_input.shape}")
print(f"SE 模块输出形状: {test_output.shape}")
print(f"SE 模块参数量: {sum(p.numel() for p in se.parameters())}")

In [ ]:
# ====== 4. MobileBlock (核心构建块) ======

class MobileBlock(nn.Module):
    """
    MobileNetV3 基础块 (类似 MobileNetV2 的 Bottleneck)
    
    结构:
        [1×1 Conv (扩展通道)] 
        → [3×3 / 5×5 Depthwise Conv (空间卷积)] 
        → [SE 模块 (可选)] 
        → [1×1 Conv (压缩通道)]
        → [残差连接 (如果 stride=1 且 输入输出通道相同)]
    """
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super(MobileBlock, self).__init__()
        self.out_channels = out_channels
        self.nonLinear = nonLinear
        self.SE = SE
        padding = (kernal_size - 1) // 2

        self.use_connect = stride == 1 and in_channels == out_channels

        if self.nonLinear == "RE":
            activation = nn.ReLU
        else:
            activation = h_swish

        # 1×1 扩展卷积
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(exp_size),
            activation(inplace=True)
        )
        # Depthwise 卷积
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernel_size=kernal_size, stride=stride, 
                      padding=padding, groups=exp_size),  # groups=exp_size → depthwise
            nn.BatchNorm2d(exp_size),
        )

        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)

        # 1×1 投影卷积
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(out_channels),
            activation(inplace=True)
        )

    def forward(self, x):
        out = self.conv(x)        # 扩展
        out = self.depth_conv(out)  # 深度可分离卷积
        if self.SE:
            out = self.squeeze_block(out)  # SE 模块
        out = self.point_conv(out)  # 投影
        if self.use_connect:
            return x + out  # 残差连接
        else:
            return out


# 测试 MobileBlock
block = MobileBlock(24, 40, kernal_size=5, stride=2, nonLinear="RE", SE=True, exp_size=72)
test_input = torch.randn(2, 24, 28, 28)
test_output = block(test_input)
print(f"MobileBlock 输入形状: {test_input.shape}")
print(f"MobileBlock 输出形状: {test_output.shape}")
print(f"MobileBlock 参数量: {sum(p.numel() for p in block.parameters())}")
print(f"使用残差连接: {block.use_connect}")
print(f"  注: stride=2 → 空间尺寸减半, 无法使用残差连接")

In [ ]:
# ====== 4.1 辅助函数与完整 MobileNetV3 模型 ======

def _make_divisible(v, divisor=8, min_value=None):
    """将通道数调整为 divisor 的整数倍，提高硬件效率"""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def _weights_init(m):
    """自定义权重初始化"""
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()


def get_model_parameters(model):
    """计算模型总参数量"""
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters


class MobileNetV3(nn.Module):
    """
    MobileNetV3 完整模型

    支持两种模式:
    - LARGE: 15 个 MobileBlock，适合精度优先场景
    - SMALL: 11 个 MobileBlock，适合极致轻量场景

    参数:
        model_mode: "LARGE" 或 "SMALL"
        num_classes: 分类数（默认 1000）
        multiplier: 宽度乘数（默认 1.0）
        dropout_rate: Dropout 比率（默认 0.0）
    """
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super(MobileNetV3, self).__init__()
        self.num_classes = num_classes

        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16],
                [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72],
                [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120],
                [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240],
                [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184],
                [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480],
                [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672],
                [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(in_channels=3, out_channels=init_conv_out, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(init_conv_out),
                h_swish(inplace=True),
            )

            self.block = []
            for in_channels, out_channels, kernal_size, stride, nonlinear, se, exp_size in layers:
                in_channels = _make_divisible(in_channels * multiplier)
                out_channels = _make_divisible(out_channels * multiplier)
                exp_size = _make_divisible(exp_size * multiplier)
                self.block.append(MobileBlock(in_channels, out_channels, kernal_size, stride, nonlinear, se, exp_size))
            self.block = nn.Sequential(*self.block)

            out_conv1_in = _make_divisible(160 * multiplier)
            out_conv1_out = _make_divisible(960 * multiplier)
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(out_conv1_in, out_conv1_out, kernel_size=1, stride=1),
                nn.BatchNorm2d(out_conv1_out),
                h_swish(inplace=True),
            )

            out_conv2_in = _make_divisible(960 * multiplier)
            out_conv2_out = _make_divisible(1280 * multiplier)
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(out_conv2_in, out_conv2_out, kernel_size=1, stride=1),
                h_swish(inplace=True),
                nn.Dropout(dropout_rate),
                nn.Conv2d(out_conv2_out, self.num_classes, kernel_size=1, stride=1),
            )

        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16],
                [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88],
                [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240],
                [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120],
                [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288],
                [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]

            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(in_channels=3, out_channels=init_conv_out, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(init_conv_out),
                h_swish(inplace=True),
            )

            self.block = []
            for in_channels, out_channels, kernal_size, stride, nonlinear, se, exp_size in layers:
                in_channels = _make_divisible(in_channels * multiplier)
                out_channels = _make_divisible(out_channels * multiplier)
                exp_size = _make_divisible(exp_size * multiplier)
                self.block.append(MobileBlock(in_channels, out_channels, kernal_size, stride, nonlinear, se, exp_size))
            self.block = nn.Sequential(*self.block)

            out_conv1_in = _make_divisible(96 * multiplier)
            out_conv1_out = _make_divisible(576 * multiplier)
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(out_conv1_in, out_conv1_out, kernel_size=1, stride=1),
                SqueezeBlock(out_conv1_out),
                nn.BatchNorm2d(out_conv1_out),
                h_swish(inplace=True),
            )

            out_conv2_in = _make_divisible(576 * multiplier)
            out_conv2_out = _make_divisible(1280 * multiplier)
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(out_conv2_in, out_conv2_out, kernel_size=1, stride=1),
                h_swish(inplace=True),
                nn.Dropout(dropout_rate),
                nn.Conv2d(out_conv2_out, self.num_classes, kernel_size=1, stride=1),
            )

        self.apply(_weights_init)

    def forward(self, x):
        out = self.init_conv(x)
        out = self.block(out)
        out = self.out_conv1(out)
        batch, channels, height, width = out.size()
        out = F.avg_pool2d(out, kernel_size=[height, width])
        out = self.out_conv2(out).view(batch, -1)
        return out


print("✅ MobileNetV3 模型定义加载完成")
print(f"  包含: h_sigmoid ✓  h_swish ✓  SqueezeBlock ✓  MobileBlock ✓  MobileNetV3 ✓")

---
## 5. 构建完整 MobileNetV3 模型

MobileNetV3 有两个版本：
- **MobileNetV3-Large**: 15 个 MobileBlock，适合高精度场景
- **MobileNetV3-Small**: 11 个 MobileBlock，适合极致轻量场景

让我们创建两个模型并对比它们的结构。

In [ ]:
# ====== 5. 创建 MobileNetV3 模型 ======
num_classes = 200  # Tiny ImageNet

# 创建 Large 和 Small 模型
model_large = MobileNetV3(model_mode="LARGE", num_classes=num_classes, multiplier=1.0, dropout_rate=0.2)
model_small = MobileNetV3(model_mode="SMALL", num_classes=num_classes, multiplier=1.0, dropout_rate=0.2)

print("=" * 60)
print("MobileNetV3-Large 参数量: ", get_model_parameters(model_large))
print("MobileNetV3-Small 参数量: ", get_model_parameters(model_small))
print("=" * 60)

In [ ]:
# ====== 6. 打印模型结构 ======
print("=" * 80)
print("MobileNetV3-Large 网络结构")
print("=" * 80)
print(model_large)

print("\n" + "=" * 80)
print("MobileNetV3-Small 网络结构")
print("=" * 80)
print(model_small)

In [ ]:
# ====== 7. 测试前向传播 ======
# 创建模拟输入: [batch_size, 3, 224, 224]
batch_size = 4
dummy_input = torch.randn(batch_size, 3, 224, 224)

# Large 模型前向传播
model_large.eval()
with torch.no_grad():
    output_large = model_large(dummy_input)
print(f"MobileNetV3-Large 输出形状: {output_large.shape}")
print(f"  期望: [{batch_size}, {num_classes}]")

# Small 模型前向传播
model_small.eval()
with torch.no_grad():
    output_small = model_small(dummy_input)
print(f"MobileNetV3-Small 输出形状: {output_small.shape}")
print(f"  期望: [{batch_size}, {num_classes}]")

---
## 6. 网络结构详细配置

MobileNetV3 的网络配置使用元组列表定义，每个元组包含：
`[in_channels, out_channels, kernel_size, stride, activation, use_se, expansion_size]`

In [ ]:
# ====== 8. 展示每层配置 ======
large_layers = [
    [16, 16, 3, 1, "RE", False, 16],
    [16, 24, 3, 2, "RE", False, 64],
    [24, 24, 3, 1, "RE", False, 72],
    [24, 40, 5, 2, "RE", True, 72],
    [40, 40, 5, 1, "RE", True, 120],
    [40, 40, 5, 1, "RE", True, 120],
    [40, 80, 3, 2, "HS", False, 240],
    [80, 80, 3, 1, "HS", False, 200],
    [80, 80, 3, 1, "HS", False, 184],
    [80, 80, 3, 1, "HS", False, 184],
    [80, 112, 3, 1, "HS", True, 480],
    [112, 112, 3, 1, "HS", True, 672],
    [112, 160, 5, 1, "HS", True, 672],
    [160, 160, 5, 2, "HS", True, 672],
    [160, 160, 5, 1, "HS", True, 960],
]

print(f"{'#':>3} | {'输入':>5} → {'输出':>5} | {'K':>1} | {'S':>1} | {'激活':>4} | {'SE':>3} | {'扩展':>4} | {'残差':>4}")
print("-" * 60)

for i, (in_c, out_c, k, s, act, se, exp) in enumerate(large_layers):
    use_residual = "✅" if (s == 1 and in_c == out_c) else "❌"
    print(f"{i+1:3d} | {in_c:>5} → {out_c:>5} | {k} | {s} | {act:>4} | {'✅' if se else '❌':>3} | {exp:>4} | {use_residual:>4}")

print("\n💡 K: 卷积核大小 | S: 步长 | SE: Squeeze-and-Excite | 扩展: 扩展层通道数")
print("💡 残差连接仅在 stride=1 且 输入=输出通道时可用")

---
## 📝 实验小结

在本实验中，我们完成了：

1. ✅ 理解了 MobileNetV3 的三大核心技术：深度可分离卷积、SE 模块、h-swish
2. ✅ 逐模块实现了网络组件：h_sigmoid → SqueezeBlock → MobileBlock → MobileNetV3
3. ✅ 可视化了 h_sigmoid / h_swish 激活函数
4. ✅ 创建了 LARGE 和 SMALL 两个模型，对比了参数量
5. ✅ 测试了前向传播的正确性

**关键数据：**

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">模型</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">参数量</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">MobileBlock 数</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">适用场景</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">MobileNetV3-Large</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">~4.0M</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">15</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">精度优先的移动端场景</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">MobileNetV3-Small</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">~1.7M</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">11</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">极致轻量场景</td>
    </tr>
  </tbody>
</table>

---


## 课后练习

1. (单选题) 深度可分离卷积中，Depthwise 卷积的 groups 应设置为？
   - A. 1
   - B. 输入通道数
   - C. 输出通道数
   - D. batch size

2. (单选题) MobileBlock 中 expansion ratio=4、输入通道 C_in=16 时，Depthwise 卷积的输入通道数是？
   - A. 16
   - B. 64
   - C. 4
   - D. 224

3. (单选题) h-swish 的表达式为 x·ReLU6(x+3)/6。当 x=6 时，输出为？
   - A. 0
   - B. 3
   - C. 6
   - D. 9

4. (多选题) MobileNetV3 降低计算量的设计包括？
   - A. 深度可分离卷积
   - B. 1×1 瓶颈投影
   - C. width multiplier
   - D. 使用更大的 7×7 卷积核

5. (多选题) SE 模块（Squeeze-and-Excitation）包含哪些操作？
   - A. 全局平均池化
   - B. 降维 FC + ReLU
   - C. 升维 FC + 激活
   - D. 对特征图逐通道缩放

6. (判断题) Squeeze 步骤使用全局平均池化将每个通道压缩为单个标量。

7. (判断题) h-swish 内部包含 sigmoid 与指数运算，部署成本与原始 swish 完全相同。

8. (填空题) MobileNetV3-Large 中 MobileBlock 的数量为 ____。

9. (填空题) width multiplier=0.75 时，通道数约为原来的 0.75 倍，参数量理论上约为原来的 ____ 倍。

10. (简答题) 为什么 MobileBlock 只有在 stride=1 且输入输出通道相同时才使用残差连接？

11. (简答题) 为什么 SE 模块通常放在 Depthwise 之后而不是 Pointwise 之前？结合感受野和通道注意力解释。

12. (代码设计题) 补全 SqueezeBlock.forward：输入 x，先全局平均池化，再两个 FC 得到通道权重，最后与 x 逐通道相乘。

13. (单选题) MobileBlock 中真正完成跨通道信息混合的是？
   - A. 3×3 Depthwise
   - B. 1×1 Pointwise
   - C. BN
   - D. h-swish

14. (多选题) 关于深度可分离卷积的正确结论包括？
   - A. 空间卷积与通道混合被解耦
   - B. Depthwise groups=输入通道数
   - C. Pointwise 负责通道混合
   - D. 相同通道数下 FLOPs 通常低于标准卷积

15. (简答题) 设 C=64、K=3、H=W=112，分别写出标准卷积和 Depthwise+Pointwise 的 FLOPs 表达式，并计算比例。

> 参考答案见 answer/02.04_model_structure_answer.ipynb。